# Healthcare Challenge 1 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 1: 30-Day Readmission Prediction**.

**Goal**: Predict `readmit_30d` (0/1) for each hospital admission
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular admission data with a simple Random Forest classifier.


In [6]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="adsb_AJp9fgtjuTRZ4Tk0mNnYDWxG_1759039055",        # Get from your team dashboard
    team_name="clai"     # Your exact team name
)


# Previous best 0.9014 Choose submission_v25_global for submission

In [2]:
# ============================================================
# Healthcare Challenge 1 — 30-Day Readmission (Macro-F1)
# v25: v21 强基线 + 决策层稳增益
#  - 不改主体建模（LGB + TF-LR + presence 注入 + NB-SVM + 词典/随访/用药/嵌入 + 等值回归校准）
#  - 改“阈值层”：Charlson(low/mid/high) × Weekend(weekday/weekend) => 6 段阈值
#  - 对 6 段阈值做“收缩(shrinkage)”向全局阈值，降低过拟合
#  - 稳健微调：LGB 多种子小集成（[42,1337]）+ TOP_K_NGRAMS=450
# 输出：
#  - submission_v25_segment6_shrink.csv  (推荐)
#  - submission_v25_segment6_plain.csv   (对照)
#  - submission_v25_global.csv           (保底)
# ============================================================

# ------------ 基本开关（基于 v21） -------------
USE_NEGATION          = True
EMBED_MODEL_PRIMARY   = "emilyalsentzer/Bio_ClinicalBERT"
EMBED_MODEL_FALLBACK  = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM_PCA         = 32
N_FOLDS               = 5
SEEDS_LGB             = [42, 1337]   # 小集成，稳抬 0.001~0.003

# TF-IDF 设置
MAX_FEAT_WORD         = 12000
MAX_FEAT_CHAR         = 14000
MIN_DF_TFIDF          = 2
LR_PENALTY            = "l2"
LR_C                  = 1.0

# n-gram presence 注入（微调到 450）
TOP_K_NGRAMS          = 450
PRES_MIN_DF           = 5

# 搜索网格（阈值更细）
ALPHAS                = [i/50 for i in range(0,51)]  # 0..1 step 0.02
THRESH_GRID           = [i/100 for i in range(5,96)] + [i/200 for i in range(70,131)]  # 0.05..0.95 + 0.35..0.65(0.005)

# 收缩强度（越大越靠近全局阈值；200 是中等偏稳的经验值）
SHRINK_K              = 200

# ------------ 路径 -------------
TRAIN_CSV = r"D:/AgentDs/agent_ds_healthcare/admissions_train.csv"
TEST_CSV  = r"D:/AgentDs/agent_ds_healthcare/admissions_test.csv"
NOTES_JSON= r"D:/AgentDs/agent_ds_healthcare/discharge_notes.json"

# ============================================================
import os, re, gc, json, math, random, warnings, io
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

try:
    import torch
    TORCH_OK = True
except Exception:
    TORCH_OK = False

try:
    import lightgbm as lgb
    LGB_OK = True
except Exception:
    LGB_OK = False
    print("Please install lightgbm: pip install lightgbm")

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# ---------------- 工具函数 ----------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    if TORCH_OK:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

def best_f1_threshold(y_true, y_prob, grid=THRESH_GRID, sample_weight=None):
    f1s = [f1_score(y_true, (y_prob>=t).astype(int), average="macro", sample_weight=sample_weight) for t in grid]
    idx = int(np.argmax(f1s))
    return float(grid[idx]), float(f1s[idx])

def sanitize_names(names):
    cleaned, used = [], {}
    for n in names:
        s = re.sub(r'[^A-Za-z0-9_]+', '_', str(n))
        s = re.sub(r'_+', '_', s).strip('_') or 'f'
        if s in used:
            used[s]+=1; s=f"{s}__{used[s]}"
        else:
            used[s]=0
        cleaned.append(s)
    return cleaned

def sanitize_df_columns(df):
    df=df.copy(); df.columns=sanitize_names(df.columns); return df

def as_weekday_int(val):
    if pd.isna(val): return np.nan
    if isinstance(val,(int,np.integer)): return int(val)
    s=str(val).strip().lower()
    mp=dict(mon=1,monday=1,tue=2,tuesday=2,wed=3,wednesday=3,thu=4,thursday=4,fri=5,friday=5,sat=6,saturday=6,sun=7,sunday=7)
    return mp.get(s,np.nan)

def load_notes_to_df(path):
    try:
        df=pd.read_json(path)
        cols=[c.lower() for c in df.columns]; df.columns=cols
        if "admission_id" in cols and any(c in cols for c in ["note","text","discharge_note"]):
            note_col="note" if "note" in cols else ("text" if "text" in cols else "discharge_note")
            out=df[["admission_id",note_col]].copy(); out.columns=["admission_id","note"]
            out["admission_id"]=pd.to_numeric(out["admission_id"],errors="coerce").astype("Int64")
            out["note"]=out["note"].astype(str); return out
        if df.shape[1]==1 and df.index.name in (None,"admission_id"):
            out=df.reset_index(); out.columns=["admission_id","note"]
            out["admission_id"]=pd.to_numeric(out["admission_id"],errors="coerce").astype("Int64")
            out["note"]=out["note"].astype(str); return out
    except Exception:
        pass
    with open(path,"r",encoding="utf-8") as f: data=json.load(f)
    if isinstance(data,dict):
        rec=[{"admission_id":int(k),"note":str(v)} for k,v in data.items()]
        out=pd.DataFrame(rec)
    else:
        tmp=pd.DataFrame(data); cols=[c.lower() for c in tmp.columns]; tmp.columns=cols
        note_col="note" if "note" in cols else ("text" if "text" in cols else "discharge_note")
        out=tmp[["admission_id",note_col]].copy(); out.columns=["admission_id","note"]
    out["admission_id"]=pd.to_numeric(out["admission_id"],errors="coerce").astype("Int64")
    out["note"]=out["note"].astype(str)
    return out

# ---- 词典（与 v21 同） ----
LEXICON = {
    "lex_fall":[r"\bfall(s)?\b", r"\bnear[-\s]?fall\b", "unsteady gait", "poor balance", "high fall risk",
                r"requires (?:supervision|assist(?:ance)?) for (?:transfers|ambulation)"],
    "lex_mobility":[r"\bwalker\b", r"\bcane\b", "wheelchair", "short distance ambulation", "limited mobility",
                    "needs assistance for transfers"],
    "lex_nutrition":["poor oral intake","reduced appetite","weight loss","cachectic","malnutrition"],
    "lex_support":["home health","caregiver","family provides care","needs prompt for medication",
                   "needs prompts for medication","medication box","medication organizer"],
    "lex_dme":["grab bars","shower chair","commode","hospital bed","raised toilet"],
    "lex_cog":["memory issues","memory impairment","confusion","disoriented","cognitive decline","cognitive impairment"],
    "lex_followup":[r"(?:pcp|primary care|family physician) follow[- ]?up","follow[- ]?up appointment"],
    "lex_living_alone":[r"\blives? alone\b","no family support","limited social support"],
}
LEXICON_EXTRA = {
    "lex_chf": ["heart failure","congestive heart failure","hfrEF","hfpEF","reduced ejection fraction","hf"],
    "lex_copd": ["copd","chronic obstructive","emphysema"],
    "lex_ckd": ["ckd","chronic kidney","end stage renal","esrd","dialysis","hemodialysis","peritoneal dialysis"],
    "lex_dm": ["diabetes","diabetic","dm2","dm type 2","dm type ii","insulin"],
    "lex_cad": ["coronary artery disease","cad","myocardial infarction","mi","stemi","nstemi"],
    "lex_cva": ["stroke","cva","tia"],
    "lex_dementia": ["dementia","alzheimer"],
    "lex_cirrhosis": ["cirrhosis","portal hypertension","esophageal varices","ascites"],
    "lex_cancer": ["cancer","carcinoma","lymphoma","leukemia","oncology","chemotherapy","radiation therapy"],
    "lex_psych": ["depression","anxiety","bipolar","schizophrenia","psychosis","suicidal"],
    "lex_substance": ["alcohol use","etoh","withdrawal","opioid use","heroin","methamphetamine","cocaine","substance use"],
    "lex_snf": ["skilled nursing facility","snf"],
    "lex_rehab": ["rehab","inpatient rehab","acute rehab"],
    "lex_home_health": ["home health","vna","visiting nurse","home care"],
    "lex_pt_ot": ["physical therapy","pt evaluation","pt/ot","occupational therapy"],
    "lex_hospice": ["hospice","palliative"],
    "lex_language": ["language barrier","interpreter"],
    "lex_transport": ["transportation barrier","no transportation","lack of transportation"],
    "lex_housing": ["homeless","shelter","housing instability","no stable housing"],
    "lex_iv_abx": ["iv antibiotics","intravenous antibiotics","opAT","picc"],
    "lex_catheter": ["foley catheter","urinary catheter","indwelling catheter"],
    "lex_ostomy": ["colostomy","ileostomy","urostomy","ostomy"],
    "lex_wound": ["wound care","wound vac","vac dressing","drain care"],
}

NEG_PREFIX=r"(?:no|denies|without|negative for|not)\s+(?:\w+\s+){0,3}"
def _compile_re(pats):
    alt="(?:"+"|".join(pats)+")"
    return re.compile(alt,re.I), re.compile(NEG_PREFIX+alt,re.I)
COMPILED={k:_compile_re(v) for k,v in LEXICON.items()}
COMPILED_EXTRA={k:_compile_re(v) for k,v in LEXICON_EXTRA.items()}

def lexicon_feats(s:pd.Series,use_negation=True)->pd.DataFrame:
    s=s.fillna("").str.lower()
    feats={}
    for k,(pos,neg) in COMPILED.items():
        cnt_pos=s.apply(lambda x: len(re.findall(pos,x)))
        if use_negation:
            cnt_neg=s.apply(lambda x: len(re.findall(neg,x))); cnt=(cnt_pos-cnt_neg).clip(lower=0)
        else:
            cnt=cnt_pos
        feats[f"{k}_cnt"]=cnt; feats[f"{k}_any"]=(cnt>0).astype(int)
    return pd.DataFrame(feats)

def lexicon_feats_extra(s:pd.Series,use_negation=True)->pd.DataFrame:
    s=s.fillna("").str.lower()
    feats={}
    for k,(pos,neg) in COMPILED_EXTRA.items():
        cnt_pos=s.apply(lambda x: len(re.findall(pos,x)))
        if use_negation:
            cnt_neg=s.apply(lambda x: len(re.findall(neg,x))); cnt=(cnt_pos-cnt_neg).clip(lower=0)
        else:
            cnt=cnt_pos
        feats[f"{k}_cnt"]=cnt; feats[f"{k}_any"]=(cnt>0).astype(int)
    return pd.DataFrame(feats)

def meds_followup_feats(s:pd.Series)->pd.DataFrame:
    s=s.fillna("").str.lower()
    freq_tokens = [" bid"," tid"," qid"," qhs"," qod"," q am","qam"," q pm","qpm"," q8h"," q12h"," q6h",
                   "daily","every","qday","once daily","twice daily","three times","four times","prn","as needed"]
    med_tokens  = [" mg"," tab"," tablet"," cap"," capsule"," inhaler"," patch"," syrup"," suspension"]
    def count_tokens(text, toks): return sum(text.count(t) for t in toks)
    day_pat = re.compile(r'\b(\d+)\s*(day|days|d)\b', re.I)
    wk_pat  = re.compile(r'\b(\d+)\s*(week|weeks|wk|wks)\b', re.I)
    def min_followup_days(text):
        nums=[]
        for m in day_pat.finditer(text): nums.append(int(m.group(1)))
        for m in wk_pat.finditer(text): nums.append(int(m.group(1))*7)
        if "tomorrow" in text: nums.append(1)
        if "next week" in text: nums.append(7)
        if len(nums)==0: return 999
        return int(min(nums))
    df = pd.DataFrame({
        "med_freq_tokens": s.apply(lambda x: count_tokens(x, freq_tokens)),
        "med_name_tokens": s.apply(lambda x: count_tokens(x, med_tokens)),
        "followup_days_min": s.apply(min_followup_days),
    })
    df["followup_days_min"] = df["followup_days_min"].clip(0, 60)
    df["followup_within_7d"] = (df["followup_days_min"] <= 7).astype(int)
    df["polypharmacy_flag"]  = (df["med_name_tokens"] >= 10).astype(int)
    df["med_freq_tokens"] = df["med_freq_tokens"].clip(0,50)
    df["med_name_tokens"] = df["med_name_tokens"].clip(0,200)
    return df

def note_stats(s:pd.Series)->pd.DataFrame:
    s=s.fillna("").astype(str)
    return pd.DataFrame({
        "note_len_chars": s.apply(len).astype(int),
        "note_len_words": s.apply(lambda x: len(x.split())).astype(int),
        "note_num_digits": s.apply(lambda x: sum(ch.isdigit() for ch in x)).astype(int),
        "note_num_neg_tokens": s.str.lower().apply(lambda x: sum(w in x for w in ["no ","denies ","without ","negative for ","not "])),
    })

# ---- 嵌入 -> PCA ----
def encode_notes(train_notes,test_notes):
    device="cuda" if (TORCH_OK and torch.cuda.is_available()) else "cpu"
    try:
        from sentence_transformers import SentenceTransformer
        model=SentenceTransformer(EMBED_MODEL_PRIMARY,device=device)
        print(f"[Emb] {EMBED_MODEL_PRIMARY} on {device}")
    except Exception:
        from sentence_transformers import SentenceTransformer
        model=SentenceTransformer(EMBED_MODEL_FALLBACK,device=device)
        print(f"[Emb] Fallback {EMBED_MODEL_FALLBACK} on {device}")
    Z_tr=model.encode(train_notes,batch_size=64,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
    Z_te=model.encode(test_notes, batch_size=64,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
    pca=PCA(n_components=EMBED_DIM_PCA,random_state=42)
    P_tr=pca.fit_transform(Z_tr); P_te=pca.transform(Z_te)
    return pd.DataFrame(P_tr,columns=[f"note_pca{i}" for i in range(P_tr.shape[1])]), \
           pd.DataFrame(P_te,columns=[f"note_pca{i}" for i in range(P_te.shape[1])])

# ---- LACE-like ----
def lace_like(df):
    los=df["los_days"].fillna(0)
    L=np.select([los<=1,los==2,los==3,(los>=4)&(los<=6),(los>=7)&(los<=13),los>=14],[1,2,3,4,5,7], default=1)
    A=df["acuity_emergent"].fillna(0).astype(int)*3
    cb=df["charlson_band"].fillna(0)
    C=np.select([cb<=0,cb==1,cb==2,cb==3,cb>=4],[0,1,2,3,5], default=0)
    ev=df["ed_visits_6m"].fillna(0)
    E=np.select([ev==0,ev==1,ev==2,ev>=3],[0,1,2,3], default=0)
    return pd.Series(L+A+C+E,name="lace_score")

# ---- LGB 表格+文本嵌入特征 ----
def assemble_lgb_features(train_df,test_df):
    for df in (train_df,test_df):
        df["discharge_weekday_int"]=df["discharge_weekday"].apply(as_weekday_int)
        df["is_weekend"]=df["discharge_weekday_int"].isin([6,7]).astype(int)
        df["is_friday"] =(df["discharge_weekday_int"]==5).astype(int)
        for c in ["los_days","ed_visits_6m","charlson_band","acuity_emergent"]:
            df[c]=pd.to_numeric(df[c],errors="coerce")
        df["los_log1p"]=np.log1p(df["los_days"].clip(lower=0))
        df["ed_visits_capped"]=df["ed_visits_6m"].clip(0,10)
        df["los_x_charlson"]=df["los_days"].fillna(0)*df["charlson_band"].fillna(0)
        df["ed_x_charlson"] =df["ed_visits_6m"].fillna(0)*df["charlson_band"].fillna(0)
        df["acuity_x_ed"]   =df["acuity_emergent"].fillna(0)*df["ed_visits_6m"].fillna(0)
    # 文本词典 / 统计 / 嵌入 / LACE
    tr_lex=lexicon_feats(train_df["note"],USE_NEGATION)
    te_lex=lexicon_feats(test_df["note"], USE_NEGATION)
    tr_lex2=lexicon_feats_extra(train_df["note"],USE_NEGATION)
    te_lex2=lexicon_feats_extra(test_df["note"], USE_NEGATION)
    tr_med=meds_followup_feats(train_df["note"])
    te_med=meds_followup_feats(test_df["note"])
    tr_stat=note_stats(train_df["note"])
    te_stat=note_stats(test_df["note"])
    P_tr,P_te=encode_notes(train_df["note"].fillna("").astype(str).tolist(),
                           test_df["note"].fillna("").astype(str).tolist())
    # 主诊断 one-hot
    for df in (train_df,test_df):
        if "primary_dx" not in df.columns: df["primary_dx"]="UNK"
    dx_all=pd.get_dummies(pd.concat([train_df["primary_dx"],test_df["primary_dx"]],axis=0).astype(str),
                          prefix="dx",drop_first=False)
    dx_tr=dx_all.iloc[:len(train_df)].reset_index(drop=True)
    dx_te=dx_all.iloc[len(train_df):].reset_index(drop=True)
    # LACE
    lace_tr=lace_like(train_df); lace_te=lace_like(test_df)
    base_cols=["los_days","los_log1p","ed_visits_6m","ed_visits_capped","charlson_band","acuity_emergent",
               "discharge_weekday_int","is_weekend","is_friday","los_x_charlson","ed_x_charlson","acuity_x_ed"]
    Xtr=pd.concat([train_df[base_cols].reset_index(drop=True),dx_tr,tr_lex.reset_index(drop=True),
                   tr_lex2.reset_index(drop=True),tr_med.reset_index(drop=True),tr_stat.reset_index(drop=True),
                   P_tr.reset_index(drop=True),lace_tr.reset_index(drop=True)],axis=1)
    Xte=pd.concat([test_df[base_cols].reset_index(drop=True), dx_te,te_lex.reset_index(drop=True),
                   te_lex2.reset_index(drop=True),te_med.reset_index(drop=True),te_stat.reset_index(drop=True),
                   P_te.reset_index(drop=True),lace_te.reset_index(drop=True)],axis=1)
    Xtr=Xtr.apply(pd.to_numeric,errors="coerce").fillna(0.0)
    Xte=Xte.apply(pd.to_numeric,errors="coerce").fillna(0.0)
    return Xtr,Xte,list(Xtr.columns)

# ---- LGB 训练/OOF ----
def lgb_params(seed, features):
    mono_pos={"los_log1p","ed_visits_capped","charlson_band","acuity_emergent",
              "los_x_charlson","ed_x_charlson","acuity_x_ed","lace_score"}
    mono=[1 if f in mono_pos else 0 for f in features]
    return dict(objective="binary", learning_rate=0.05, num_leaves=63, max_depth=6,
                min_data_in_leaf=120, feature_fraction=0.80, lambda_l2=5.0,
                n_estimators=2200, subsample=0.9, subsample_freq=1,
                random_state=seed, class_weight="balanced",
                n_jobs=os.cpu_count() or 4, monotone_constraints=mono, verbose=-1)

def cv_oof_lgb(X,y,groups,features):
    gkf=GroupKFold(n_splits=N_FOLDS)
    oof=np.zeros(len(y),dtype=float)
    for seed in SEEDS_LGB:
        set_seed(seed)
        for fold,(tr,va) in enumerate(gkf.split(X,y,groups=groups),1):
            clf=lgb.LGBMClassifier(**lgb_params(seed,features))
            clf.fit(X.iloc[tr],y[tr],
                    eval_set=[(X.iloc[va],y[va])],
                    eval_metric="auc",
                    callbacks=[lgb.early_stopping(stopping_rounds=200,verbose=False)])
            oof[va]+=clf.predict_proba(X.iloc[va])[:,1]
    oof/=len(SEEDS_LGB)
    return oof

def fit_lgb_full(X,y,Xte,features):
    prob=np.zeros(len(Xte),dtype=float)
    for seed in SEEDS_LGB:
        set_seed(seed)
        clf=lgb.LGBMClassifier(**lgb_params(seed,features))
        clf.fit(X,y)
        prob+=clf.predict_proba(Xte)[:,1]
    return prob/len(SEEDS_LGB)

# ---- TF-IDF（word+char） ----
def build_tfidf(train_texts,test_texts):
    v_word=TfidfVectorizer(ngram_range=(1,2),max_features=MAX_FEAT_WORD,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    v_char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),max_features=MAX_FEAT_CHAR,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    Xw_tr=v_word.fit_transform(train_texts); Xw_te=v_word.transform(test_texts)
    Xc_tr=v_char.fit_transform(train_texts); Xc_te=v_char.transform(test_texts)
    Xtr=hstack([Xw_tr,Xc_tr]); Xte=hstack([Xw_te,Xc_te])
    names_word=[f"w_{t}" for t in v_word.get_feature_names_out()]
    names_char=[f"c_{t}" for t in v_char.get_feature_names_out()]
    feat_names=sanitize_names([*names_word,*names_char])
    return Xtr,Xte,feat_names

def cv_oof_tfidf_lr(X_sparse,y,groups,C=1.0,penalty="l2"):
    gkf=GroupKFold(n_splits=N_FOLDS)
    oof=np.zeros(len(y),dtype=float)
    for fold,(tr,va) in enumerate(gkf.split(np.arange(X_sparse.shape[0]),y,groups=groups),1):
        clf=LogisticRegression(solver="saga",penalty=penalty,C=C,max_iter=500,
                               class_weight="balanced",n_jobs=os.cpu_count() or 4)
        clf.fit(X_sparse[tr],y[tr])
        oof[va]=clf.predict_proba(X_sparse[va])[:,1]
    return oof

def fit_tfidf_lr_full(X_sparse,y,X_sparse_te,C=1.0,penalty="l2"):
    clf=LogisticRegression(solver="saga",penalty=penalty,C=C,max_iter=600,
                           class_weight="balanced",n_jobs=os.cpu_count() or 4)
    clf.fit(X_sparse,y)
    probs=clf.predict_proba(X_sparse_te)[:,1]
    return clf, probs

# ---- NB-SVM log-count ratio 分数（3 列），注入 LGB 特征 ----
def nbsvm_scores(X_tr, X_te, y):
    pos = y==1; neg = ~pos
    count_pos = 1 + X_tr[pos].sum(axis=0)
    count_neg = 1 + X_tr[neg].sum(axis=0)
    p_pos = np.asarray(count_pos / count_pos.sum()).ravel()
    p_neg = np.asarray(count_neg / count_neg.sum()).ravel()
    r = np.log((p_pos + 1e-12) / (p_neg + 1e-12))
    s_tr = np.asarray(X_tr.dot(r)).ravel()
    s_te = np.asarray(X_te.dot(r)).ravel()
    return s_tr, s_te

# ---- presence 注入 ----
def topk_presence_from_lr(clf, X_tr, X_te, feat_names, k=400, min_df=5):
    coef = np.ravel(clf.coef_)
    idx_sorted = np.argsort(np.abs(coef))[::-1]
    df_counts = np.array((X_tr[:, idx_sorted] > 0).sum(axis=0)).ravel()
    keep_mask = df_counts >= min_df
    idx_filtered = idx_sorted[keep_mask][:k]
    selected = [feat_names[i] for i in idx_filtered]
    tr_bin = (X_tr[:, idx_filtered] > 0).astype(np.int8).toarray()
    te_bin = (X_te[:, idx_filtered] > 0).astype(np.int8).toarray()
    colnames = [f"ngram_top_{j}_{nm}" for j, nm in enumerate(sanitize_names(selected))]
    df_tr = pd.DataFrame(tr_bin, columns=colnames)
    df_te = pd.DataFrame(te_bin, columns=colnames)
    return df_tr, df_te, selected

# ---- 等值回归校准 ----
def isotonic_fit_transform(y_true, oof_prob, test_prob):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_prob, y_true)
    return iso.transform(oof_prob), iso.transform(test_prob), iso

# ---- 融合 ----
def prob_blend(p1, p2, alpha): return alpha*p1 + (1-alpha)*p2

# ---- 6 段: Charlson(low/mid/high) × Weekend(0/1) ----
def segment6_keys(cb, wkend):
    cb = pd.Series(cb).fillna(0)
    band = np.where(cb<=1,"low",np.where(cb>=4,"high","mid"))
    wk = np.where(pd.Series(wkend).astype(int)==1, "wknd", "wkdy")
    return np.char.add(np.char.add(band.astype(str), "_"), wk.astype(str))  # e.g., 'low_wkdy'

def segmented_thresholds_6(y, p, seg_keys, grid=THRESH_GRID, sample_weight=None):
    thr = {}; pred = np.zeros_like(y, dtype=int); counts={}
    uniq = np.unique(seg_keys)
    # 先求全局阈值（用于收缩）
    thr_g,_ = best_f1_threshold(y, p, grid, sample_weight=sample_weight)
    for seg in uniq:
        idx = (seg_keys==seg)
        counts[seg] = int(idx.sum())
        if counts[seg]==0:
            thr[seg]=thr_g; continue
        w = None if sample_weight is None else np.asarray(sample_weight)[idx]
        t_seg,_ = best_f1_threshold(y[idx], p[idx], grid, sample_weight=w)
        thr[seg]=t_seg
        pred[idx] = (p[idx] >= t_seg).astype(int)
    f1 = f1_score(y, pred, average="macro", sample_weight=sample_weight)
    return thr, thr_g, counts, f1, pred

def shrink_thresholds(thr_map, counts_map, thr_global, k=200):
    # James–Stein 风格收缩：t' = (n/(n+k))*t + (k/(n+k))*t_global
    out={}
    for seg,t in thr_map.items():
        n = counts_map.get(seg,0)
        w = n/(n+k)
        out[seg] = float(w*t + (1-w)*thr_global)
    return out

def apply_segment_thresholds(p, seg_keys, thr_map, default_thr=0.5):
    pred = np.zeros(len(p), dtype=int)
    for seg,thr in thr_map.items():
        idx = (seg_keys==seg); 
        if idx.sum()>0:
            pred[idx] = (p[idx] >= thr).astype(int)
    # 若有没覆盖的 seg（理论上不会），用默认阈值
    miss = ~(np.isin(seg_keys, list(thr_map.keys())))
    if miss.any():
        pred[miss] = (p[miss] >= default_thr).astype(int)
    return pred

def search_alpha_segment6_with_shrink(y, p1, p2, seg_keys, alphas=ALPHAS, grid=THRESH_GRID, k=SHRINK_K, sample_weight=None):
    best={"alpha":0.5,"thr_map":{}, "thr_map_shrink":{}, "thr_global":0.5, "f1_plain":-1.0, "f1_shrink":-1.0}
    for a in alphas:
        mix = prob_blend(p1,p2,a)
        thr_map, thr_g, counts, f1_plain, _ = segmented_thresholds_6(y, mix, seg_keys, grid, sample_weight)
        thr_map_shrink = shrink_thresholds(thr_map, counts, thr_g, k=k)
        # 计算收缩后的 OOF F1
        pred_sh = apply_segment_thresholds(mix, seg_keys, thr_map_shrink, default_thr=thr_g)
        f1_sh = f1_score(y, pred_sh, average="macro", sample_weight=sample_weight)
        if f1_sh > best["f1_shrink"]:
            best={"alpha":float(a),"thr_map":thr_map,"thr_map_shrink":thr_map_shrink,
                  "thr_global":float(thr_g),"f1_plain":float(f1_plain),"f1_shrink":float(f1_sh)}
    return best

# ============================ 主流程 ============================
def main():
    # 1) 读取 & 合并
    train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); notes=load_notes_to_df(NOTES_JSON)
    for df in (train,test,notes): df.columns=[c.strip() for c in df.columns]
    train=train.merge(notes,on="admission_id",how="left")
    test =test.merge(notes, on="admission_id",how="left")
    assert "readmit_30d" in train.columns and "patient_id" in train.columns
    y=train["readmit_30d"].astype(int).values
    groups=train["patient_id"].astype(str).values

    # 2) LGB 特征（与 v21 一致）
    Xtr_lgb, Xte_lgb, feat_names = assemble_lgb_features(train.copy(), test.copy())
    print(f"[LGB] base X_train={Xtr_lgb.shape}, X_test={Xte_lgb.shape}, features={len(feat_names)}")

    # 3) TF-IDF（word+char 组合）
    texts_tr=train["note"].fillna("").astype(str).tolist()
    texts_te=test["note"].fillna("").astype(str).tolist()
    Xtr_tfidf, Xte_tfidf, tfidf_feat_names = build_tfidf(texts_tr, texts_te)

    # 4) NB-SVM 分数（3 列）：word、char、combined -> 注入到 LGB 特征
    # 为了严格贴近 v21，这里分别再建 word/char（代价很小，相对嵌入）
    v_word=TfidfVectorizer(ngram_range=(1,2),max_features=MAX_FEAT_WORD,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    v_char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),max_features=MAX_FEAT_CHAR,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    Xw_tr=v_word.fit_transform(texts_tr); Xw_te=v_word.transform(texts_te)
    Xc_tr=v_char.fit_transform(texts_tr); Xc_te=v_char.transform(texts_te)
    def _nb(Xtr, Xte, y):
        pos = y==1; neg = ~pos
        count_pos = 1 + Xtr[pos].sum(axis=0)
        count_neg = 1 + Xtr[neg].sum(axis=0)
        p_pos = np.asarray(count_pos / count_pos.sum()).ravel()
        p_neg = np.asarray(count_neg / count_neg.sum()).ravel()
        r = np.log((p_pos + 1e-12) / (p_neg + 1e-12))
        return np.asarray(Xtr.dot(r)).ravel(), np.asarray(Xte.dot(r)).ravel()
    nbw_tr, nbw_te = _nb(Xw_tr, Xw_te, y)
    nbc_tr, nbc_te = _nb(Xc_tr, Xc_te, y)
    nba_tr, nba_te = _nb(Xtr_tfidf, Xte_tfidf, y)
    Xtr_lgb["nbsvm_word"] = nbw_tr; Xte_lgb["nbsvm_word"] = nbw_te
    Xtr_lgb["nbsvm_char"] = nbc_tr; Xte_lgb["nbsvm_char"] = nbc_te
    Xtr_lgb["nbsvm_all"]  = nba_tr; Xte_lgb["nbsvm_all"]  = nba_te

    # 5) TF-LR（与 v21 一致）
    print("[TF] OOF TF-IDF(LogReg L2, C=1.0) ...")
    oof_tfidf = cv_oof_tfidf_lr(Xtr_tfidf, y, groups, C=LR_C, penalty=LR_PENALTY)
    tfidf_clf_full, pte_tfidf_raw = fit_tfidf_lr_full(Xtr_tfidf, y, Xte_tfidf, C=LR_C, penalty=LR_PENALTY)

    # 6) presence 注入（k=450）
    ngram_tr, ngram_te, sel_names = topk_presence_from_lr(tfidf_clf_full, Xtr_tfidf, Xte_tfidf,
                                                          tfidf_feat_names, k=TOP_K_NGRAMS, min_df=PRES_MIN_DF)
    Xtr_lgb_aug = pd.concat([Xtr_lgb.reset_index(drop=True), ngram_tr.reset_index(drop=True)], axis=1)
    Xte_lgb_aug = pd.concat([Xte_lgb.reset_index(drop=True), ngram_te.reset_index(drop=True)], axis=1)
    Xtr_lgb_aug = sanitize_df_columns(Xtr_lgb_aug)
    Xte_lgb_aug.columns = Xtr_lgb_aug.columns.tolist()
    feat_names_aug = Xtr_lgb_aug.columns.tolist()
    print(f"[LGB+Ngram] X_train={Xtr_lgb_aug.shape}, X_test={Xte_lgb_aug.shape}")

    # 7) LGB OOF & TEST 概率（多种子小集成）
    print("[CV] OOF LGB(AUG) ...")
    oof_lgb = cv_oof_lgb(Xtr_lgb_aug, y, groups, feat_names_aug)
    pte_lgb_raw = fit_lgb_full(Xtr_lgb_aug, y, Xte_lgb_aug, feat_names_aug)

    # 8) 两模型做 isotonic 校准
    print("[Calib] Isotonic for LGB & TF ...")
    oof_lgb_cal, pte_lgb_cal, iso_lgb = isotonic_fit_transform(y, oof_lgb, pte_lgb_raw)
    oof_tfidf_cal, pte_tfidf_cal, iso_tf = isotonic_fit_transform(y, oof_tfidf, pte_tfidf_raw)

    # 9) 在 OOF 上优化融合 + 阈值（全局 & 6 段，不带权重）
    # 9.1 先找全局最优（作为对照 & 收缩锚点）
    best_global = {"alpha":0.5,"thr":0.5,"f1":-1.0}
    for a in ALPHAS:
        mix = prob_blend(oof_lgb_cal, oof_tfidf_cal, a)
        t, f1 = best_f1_threshold(y, mix, THRESH_GRID)
        if f1 > best_global["f1"]:
            best_global = {"alpha":float(a), "thr":float(t), "f1":float(f1)}

    # 9.2 6 段（不收缩）与 6 段（收缩）共同搜索 alpha，返回两种评分做对比
    seg6_tr = segment6_keys(Xtr_lgb_aug["charlson_band"], Xtr_lgb_aug["is_weekend"])
    best_seg6 = search_alpha_segment6_with_shrink(
        y, oof_lgb_cal, oof_tfidf_cal, seg6_tr,
        alphas=ALPHAS, grid=THRESH_GRID, k=SHRINK_K, sample_weight=None
    )

    print(f"[Global] alpha={best_global['alpha']:.2f}, thr={best_global['thr']:.3f}, OOF Macro-F1={best_global['f1']:.4f}")
    print(f"[Seg6-plain]  alpha={best_seg6['alpha']:.2f}, OOF Macro-F1={best_seg6['f1_plain']:.4f}")
    print(f"[Seg6-shrink] alpha={best_seg6['alpha']:.2f}, OOF Macro-F1={best_seg6['f1_shrink']:.4f}")
    print(f"              thr_global={best_seg6['thr_global']:.3f}")
    print(f"              thr_map(shrink)={best_seg6['thr_map_shrink']}")

    # 10) 生成三份提交
    # 10.1 全局
    pte_blend_g = prob_blend(pte_lgb_cal, pte_tfidf_cal, best_global["alpha"])
    pred_global = (pte_blend_g >= best_global["thr"]).astype(int)
    sub_g = test[["admission_id"]].copy(); sub_g["readmit_30d"] = pred_global
    sub_g.to_csv("submission_v25_global.csv", index=False)

    # 10.2 6 段（不收缩）
    pte_blend_s = prob_blend(pte_lgb_cal, pte_tfidf_cal, best_seg6["alpha"])
    seg6_te = segment6_keys(Xte_lgb_aug["charlson_band"], Xte_lgb_aug["is_weekend"])
    pred_seg_plain = apply_segment_thresholds(pte_blend_s, seg6_te, best_seg6["thr_map"], default_thr=best_seg6["thr_global"])
    sub_sp = test[["admission_id"]].copy(); sub_sp["readmit_30d"] = pred_seg_plain
    sub_sp.to_csv("submission_v25_segment6_plain.csv", index=False)

    # 10.3 6 段（收缩）
    pred_seg_sh = apply_segment_thresholds(pte_blend_s, seg6_te, best_seg6["thr_map_shrink"], default_thr=best_seg6["thr_global"])
    sub_ss = test[["admission_id"]].copy(); sub_ss["readmit_30d"] = pred_seg_sh
    sub_ss.to_csv("submission_v25_segment6_shrink.csv", index=False)

    # 11) 简要 OOF 报告（6 段收缩）
    mix_oof = prob_blend(oof_lgb_cal, oof_tfidf_cal, best_seg6["alpha"])
    y_pred_oof_sh = apply_segment_thresholds(mix_oof, seg6_tr, best_seg6["thr_map_shrink"], default_thr=best_seg6["thr_global"])
    print("\n=== OOF Classification Report (Seg6-shrink) ===")
    print(classification_report(y, y_pred_oof_sh, digits=4))

    print("Saved:")
    print(" - submission_v25_segment6_shrink.csv")
    print(" - submission_v25_segment6_plain.csv")
    print(" - submission_v25_global.csv")

if __name__ == "__main__":
    if not LGB_OK:
        raise ImportError("LightGBM required. Install: pip install lightgbm")
    main()


No sentence-transformers model found with name emilyalsentzer/Bio_ClinicalBERT. Creating a new one with mean pooling.


[Emb] emilyalsentzer/Bio_ClinicalBERT on cuda


Batches: 100%|██████████| 79/79 [00:08<00:00,  9.87it/s]


[LGB] base X_train=(5000, 119), X_test=(5000, 119), features=119
[TF] OOF TF-IDF(LogReg L2, C=1.0) ...
[LGB+Ngram] X_train=(5000, 572), X_test=(5000, 572)
[CV] OOF LGB(AUG) ...
[Calib] Isotonic for LGB & TF ...
[Global] alpha=0.72, thr=0.470, OOF Macro-F1=0.9062
[Seg6-plain]  alpha=0.90, OOF Macro-F1=0.9084
[Seg6-shrink] alpha=0.90, OOF Macro-F1=0.9072
              thr_global=0.555
              thr_map(shrink)={np.str_('high_wkdy'): 0.6019686800894855, np.str_('high_wknd'): 0.4123961661341853, np.str_('low_wkdy'): 0.5770777323202806, np.str_('low_wknd'): 0.4842966751918159, np.str_('mid_wkdy'): 0.3791929133858268, np.str_('mid_wknd'): 0.574535519125683}

=== OOF Classification Report (Seg6-shrink) ===
              precision    recall  f1-score   support

           0     0.9168    0.8939    0.9052      2479
           1     0.8982    0.9203    0.9091      2521

    accuracy                         0.9072      5000
   macro avg     0.9075    0.9071    0.9072      5000
weighted avg   

# Final Best 0.9058
- 0.9058 (submission_v25_segment6_shrink)
- 0.9058 (submission_v25_segment6_plain)
- 0.9058 (submission_v25_global)

In [4]:
PATIENTS_CSV = r"D:/AgentDs/agent_ds_healthcare/patients.csv"

USE_PATIENT_TABLE   = True
USE_PATIENT_HISTORY = True

# （冲榜向）target encoding：对 patient_id / zip3 做平滑编码
USE_TARGET_ENCODING = True
TE_SMOOTH_PATIENT   = 50
TE_SMOOTH_ZIP3      = 200
TE_FOLDS            = 5
TE_SEED             = 42
# ============================================================
# Healthcare Challenge 1 — 30-Day Readmission (Macro-F1)
# v25: v21 强基线 + 决策层稳增益
#  - 不改主体建模（LGB + TF-LR + presence 注入 + NB-SVM + 词典/随访/用药/嵌入 + 等值回归校准）
#  - 改“阈值层”：Charlson(low/mid/high) × Weekend(weekday/weekend) => 6 段阈值
#  - 对 6 段阈值做“收缩(shrinkage)”向全局阈值，降低过拟合
#  - 稳健微调：LGB 多种子小集成（[42,1337]）+ TOP_K_NGRAMS=450
# 输出：
#  - submission_v25_segment6_shrink.csv  (推荐)
#  - submission_v25_segment6_plain.csv   (对照)
#  - submission_v25_global.csv           (保底)
# ============================================================

# ------------ 基本开关（基于 v21） -------------
USE_NEGATION          = True
EMBED_MODEL_PRIMARY   = "emilyalsentzer/Bio_ClinicalBERT"
EMBED_MODEL_FALLBACK  = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM_PCA         = 32
N_FOLDS               = 5
SEEDS_LGB             = [42, 1337]   # 小集成，稳抬 0.001~0.003

# TF-IDF 设置
MAX_FEAT_WORD         = 12000
MAX_FEAT_CHAR         = 14000
MIN_DF_TFIDF          = 2
LR_PENALTY            = "l2"
LR_C                  = 1.0

# n-gram presence 注入（微调到 450）
TOP_K_NGRAMS          = 450
PRES_MIN_DF           = 5

# 搜索网格（阈值更细）
ALPHAS                = [i/50 for i in range(0,51)]  # 0..1 step 0.02
THRESH_GRID           = [i/100 for i in range(5,96)] + [i/200 for i in range(70,131)]  # 0.05..0.95 + 0.35..0.65(0.005)

# 收缩强度（越大越靠近全局阈值；200 是中等偏稳的经验值）
SHRINK_K              = 200

# ------------ 路径 -------------
TRAIN_CSV = r"D:/AgentDs/agent_ds_healthcare/admissions_train.csv"
TEST_CSV  = r"D:/AgentDs/agent_ds_healthcare/admissions_test.csv"
NOTES_JSON= r"D:/AgentDs/agent_ds_healthcare/discharge_notes.json"

# ============================================================
import os, re, gc, json, math, random, warnings, io
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

try:
    import torch
    TORCH_OK = True
except Exception:
    TORCH_OK = False

try:
    import lightgbm as lgb
    LGB_OK = True
except Exception:
    LGB_OK = False
    print("Please install lightgbm: pip install lightgbm")

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# ---------------- 工具函数 ----------------
def add_patients(train_df, test_df, patients_path=PATIENTS_CSV):
    pat = pd.read_csv(patients_path)
    pat["zip3"] = pat["zip3"].astype(str)
    for df in (train_df, test_df):
        df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

    train_df = train_df.merge(pat, on="patient_id", how="left")
    test_df  = test_df.merge(pat, on="patient_id", how="left")

    for df in (train_df, test_df):
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df["sex"] = df["sex"].fillna("UNK").astype(str)
        df["insurance"] = df["insurance"].fillna("UNK").astype(str)
        df["zip3"] = df["zip3"].fillna("UNK").astype(str)
        df["age_bucket"] = pd.cut(
            df["age"], bins=[-1, 39, 64, 79, 200],
            labels=["0_39","40_64","65_79","80p"]
        ).astype(str)

    return train_df, test_df

def add_patient_history_features(train_df, test_df):
    all_df = pd.concat([train_df.assign(_is_train=1), test_df.assign(_is_train=0)], ignore_index=True)

    # 先准备 note_len
    all_df["note_len_chars"] = all_df["note"].fillna("").astype(str).apply(len)

    g = all_df.groupby("patient_id", sort=False)

    # 基础聚合（不含标签）
    agg = g.agg(
        pat_adm_count_all=("admission_id","count"),
        pat_los_mean=("los_days","mean"),
        pat_los_std=("los_days","std"),
        pat_charlson_mean=("charlson_band","mean"),
        pat_charlson_max=("charlson_band","max"),
        pat_ed_mean=("ed_visits_6m","mean"),
        pat_ed_max=("ed_visits_6m","max"),
        pat_note_len_mean=("note_len_chars","mean"),
    ).reset_index()

    all_df = all_df.merge(agg, on="patient_id", how="left")

    # primary_dx 计数（3类很适合直接展开）
    dx_cnt = pd.crosstab(all_df["patient_id"], all_df["primary_dx"])
    dx_cnt.columns = [f"pat_dx_cnt_{c}" for c in dx_cnt.columns.astype(str)]
    dx_cnt = dx_cnt.reset_index()
    all_df = all_df.merge(dx_cnt, on="patient_id", how="left")

    # leave-one-out mean（可选但常有用）
    for col in ["los_days","charlson_band","ed_visits_6m","note_len_chars"]:
        s = pd.to_numeric(all_df[col], errors="coerce").fillna(0.0)
        cnt = g[col].transform("count")
        sm  = g[col].transform("sum")
        loo = (sm - s) / (cnt - 1)
        loo = loo.where(cnt > 1, s.mean())  # n=1 回退全局均值
        all_df[f"pat_{col}_mean_loo"] = loo

    train_out = all_df[all_df["_is_train"]==1].drop(columns=["_is_train"])
    test_out  = all_df[all_df["_is_train"]==0].drop(columns=["_is_train"])
    return train_out.reset_index(drop=True), test_out.reset_index(drop=True)

from sklearn.model_selection import StratifiedKFold

def kfold_target_encode(train_df, test_df, col, y, n_splits=5, seed=42, smooth=50):
    prior = float(np.mean(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    tr_enc = np.zeros(len(train_df), dtype=float)

    tmp = train_df[[col]].copy()
    tmp["y"] = y

    for tr_idx, va_idx in skf.split(tmp, y):
        stats = tmp.iloc[tr_idx].groupby(col)["y"].agg(["count","mean"])
        enc = (stats["count"]*stats["mean"] + smooth*prior) / (stats["count"] + smooth)
        tr_enc[va_idx] = tmp.iloc[va_idx][col].map(enc).fillna(prior).values

    # test 用全量 train 拟合
    stats = tmp.groupby(col)["y"].agg(["count","mean"])
    enc = (stats["count"]*stats["mean"] + smooth*prior) / (stats["count"] + smooth)
    te_enc = test_df[col].map(enc).fillna(prior).values

    return tr_enc, te_enc

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    if TORCH_OK:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

def best_f1_threshold(y_true, y_prob, grid=THRESH_GRID, sample_weight=None):
    f1s = [f1_score(y_true, (y_prob>=t).astype(int), average="macro", sample_weight=sample_weight) for t in grid]
    idx = int(np.argmax(f1s))
    return float(grid[idx]), float(f1s[idx])

def sanitize_names(names):
    cleaned, used = [], {}
    for n in names:
        s = re.sub(r'[^A-Za-z0-9_]+', '_', str(n))
        s = re.sub(r'_+', '_', s).strip('_') or 'f'
        if s in used:
            used[s]+=1; s=f"{s}__{used[s]}"
        else:
            used[s]=0
        cleaned.append(s)
    return cleaned

def sanitize_df_columns(df):
    df=df.copy(); df.columns=sanitize_names(df.columns); return df

def as_weekday_int(val):
    if pd.isna(val): return np.nan
    if isinstance(val,(int,np.integer)): return int(val)
    s=str(val).strip().lower()
    mp=dict(mon=1,monday=1,tue=2,tuesday=2,wed=3,wednesday=3,thu=4,thursday=4,fri=5,friday=5,sat=6,saturday=6,sun=7,sunday=7)
    return mp.get(s,np.nan)

def load_notes_to_df(path):
    try:
        df=pd.read_json(path)
        cols=[c.lower() for c in df.columns]; df.columns=cols
        if "admission_id" in cols and any(c in cols for c in ["note","text","discharge_note"]):
            note_col="note" if "note" in cols else ("text" if "text" in cols else "discharge_note")
            out=df[["admission_id",note_col]].copy(); out.columns=["admission_id","note"]
            out["admission_id"]=pd.to_numeric(out["admission_id"],errors="coerce").astype("Int64")
            out["note"]=out["note"].astype(str); return out
        if df.shape[1]==1 and df.index.name in (None,"admission_id"):
            out=df.reset_index(); out.columns=["admission_id","note"]
            out["admission_id"]=pd.to_numeric(out["admission_id"],errors="coerce").astype("Int64")
            out["note"]=out["note"].astype(str); return out
    except Exception:
        pass
    with open(path,"r",encoding="utf-8") as f: data=json.load(f)
    if isinstance(data,dict):
        rec=[{"admission_id":int(k),"note":str(v)} for k,v in data.items()]
        out=pd.DataFrame(rec)
    else:
        tmp=pd.DataFrame(data); cols=[c.lower() for c in tmp.columns]; tmp.columns=cols
        note_col="note" if "note" in cols else ("text" if "text" in cols else "discharge_note")
        out=tmp[["admission_id",note_col]].copy(); out.columns=["admission_id","note"]
    out["admission_id"]=pd.to_numeric(out["admission_id"],errors="coerce").astype("Int64")
    out["note"]=out["note"].astype(str)
    return out

# ---- 词典（与 v21 同） ----
LEXICON = {
    "lex_fall":[r"\bfall(s)?\b", r"\bnear[-\s]?fall\b", "unsteady gait", "poor balance", "high fall risk",
                r"requires (?:supervision|assist(?:ance)?) for (?:transfers|ambulation)"],
    "lex_mobility":[r"\bwalker\b", r"\bcane\b", "wheelchair", "short distance ambulation", "limited mobility",
                    "needs assistance for transfers"],
    "lex_nutrition":["poor oral intake","reduced appetite","weight loss","cachectic","malnutrition"],
    "lex_support":["home health","caregiver","family provides care","needs prompt for medication",
                   "needs prompts for medication","medication box","medication organizer"],
    "lex_dme":["grab bars","shower chair","commode","hospital bed","raised toilet"],
    "lex_cog":["memory issues","memory impairment","confusion","disoriented","cognitive decline","cognitive impairment"],
    "lex_followup":[r"(?:pcp|primary care|family physician) follow[- ]?up","follow[- ]?up appointment"],
    "lex_living_alone":[r"\blives? alone\b","no family support","limited social support"],
}
LEXICON_EXTRA = {
    "lex_chf": ["heart failure","congestive heart failure","hfrEF","hfpEF","reduced ejection fraction","hf"],
    "lex_copd": ["copd","chronic obstructive","emphysema"],
    "lex_ckd": ["ckd","chronic kidney","end stage renal","esrd","dialysis","hemodialysis","peritoneal dialysis"],
    "lex_dm": ["diabetes","diabetic","dm2","dm type 2","dm type ii","insulin"],
    "lex_cad": ["coronary artery disease","cad","myocardial infarction","mi","stemi","nstemi"],
    "lex_cva": ["stroke","cva","tia"],
    "lex_dementia": ["dementia","alzheimer"],
    "lex_cirrhosis": ["cirrhosis","portal hypertension","esophageal varices","ascites"],
    "lex_cancer": ["cancer","carcinoma","lymphoma","leukemia","oncology","chemotherapy","radiation therapy"],
    "lex_psych": ["depression","anxiety","bipolar","schizophrenia","psychosis","suicidal"],
    "lex_substance": ["alcohol use","etoh","withdrawal","opioid use","heroin","methamphetamine","cocaine","substance use"],
    "lex_snf": ["skilled nursing facility","snf"],
    "lex_rehab": ["rehab","inpatient rehab","acute rehab"],
    "lex_home_health": ["home health","vna","visiting nurse","home care"],
    "lex_pt_ot": ["physical therapy","pt evaluation","pt/ot","occupational therapy"],
    "lex_hospice": ["hospice","palliative"],
    "lex_language": ["language barrier","interpreter"],
    "lex_transport": ["transportation barrier","no transportation","lack of transportation"],
    "lex_housing": ["homeless","shelter","housing instability","no stable housing"],
    "lex_iv_abx": ["iv antibiotics","intravenous antibiotics","opAT","picc"],
    "lex_catheter": ["foley catheter","urinary catheter","indwelling catheter"],
    "lex_ostomy": ["colostomy","ileostomy","urostomy","ostomy"],
    "lex_wound": ["wound care","wound vac","vac dressing","drain care"],
}

NEG_PREFIX=r"(?:no|denies|without|negative for|not)\s+(?:\w+\s+){0,3}"
def _compile_re(pats):
    alt="(?:"+"|".join(pats)+")"
    return re.compile(alt,re.I), re.compile(NEG_PREFIX+alt,re.I)
COMPILED={k:_compile_re(v) for k,v in LEXICON.items()}
COMPILED_EXTRA={k:_compile_re(v) for k,v in LEXICON_EXTRA.items()}

def lexicon_feats(s:pd.Series,use_negation=True)->pd.DataFrame:
    s=s.fillna("").str.lower()
    feats={}
    for k,(pos,neg) in COMPILED.items():
        cnt_pos=s.apply(lambda x: len(re.findall(pos,x)))
        if use_negation:
            cnt_neg=s.apply(lambda x: len(re.findall(neg,x))); cnt=(cnt_pos-cnt_neg).clip(lower=0)
        else:
            cnt=cnt_pos
        feats[f"{k}_cnt"]=cnt; feats[f"{k}_any"]=(cnt>0).astype(int)
    return pd.DataFrame(feats)

def lexicon_feats_extra(s:pd.Series,use_negation=True)->pd.DataFrame:
    s=s.fillna("").str.lower()
    feats={}
    for k,(pos,neg) in COMPILED_EXTRA.items():
        cnt_pos=s.apply(lambda x: len(re.findall(pos,x)))
        if use_negation:
            cnt_neg=s.apply(lambda x: len(re.findall(neg,x))); cnt=(cnt_pos-cnt_neg).clip(lower=0)
        else:
            cnt=cnt_pos
        feats[f"{k}_cnt"]=cnt; feats[f"{k}_any"]=(cnt>0).astype(int)
    return pd.DataFrame(feats)

def meds_followup_feats(s:pd.Series)->pd.DataFrame:
    s=s.fillna("").str.lower()
    freq_tokens = [" bid"," tid"," qid"," qhs"," qod"," q am","qam"," q pm","qpm"," q8h"," q12h"," q6h",
                   "daily","every","qday","once daily","twice daily","three times","four times","prn","as needed"]
    med_tokens  = [" mg"," tab"," tablet"," cap"," capsule"," inhaler"," patch"," syrup"," suspension"]
    def count_tokens(text, toks): return sum(text.count(t) for t in toks)
    day_pat = re.compile(r'\b(\d+)\s*(day|days|d)\b', re.I)
    wk_pat  = re.compile(r'\b(\d+)\s*(week|weeks|wk|wks)\b', re.I)
    def min_followup_days(text):
        nums=[]
        for m in day_pat.finditer(text): nums.append(int(m.group(1)))
        for m in wk_pat.finditer(text): nums.append(int(m.group(1))*7)
        if "tomorrow" in text: nums.append(1)
        if "next week" in text: nums.append(7)
        if len(nums)==0: return 999
        return int(min(nums))
    df = pd.DataFrame({
        "med_freq_tokens": s.apply(lambda x: count_tokens(x, freq_tokens)),
        "med_name_tokens": s.apply(lambda x: count_tokens(x, med_tokens)),
        "followup_days_min": s.apply(min_followup_days),
    })
    df["followup_days_min"] = df["followup_days_min"].clip(0, 60)
    df["followup_within_7d"] = (df["followup_days_min"] <= 7).astype(int)
    df["polypharmacy_flag"]  = (df["med_name_tokens"] >= 10).astype(int)
    df["med_freq_tokens"] = df["med_freq_tokens"].clip(0,50)
    df["med_name_tokens"] = df["med_name_tokens"].clip(0,200)
    return df

def note_stats(s:pd.Series)->pd.DataFrame:
    s=s.fillna("").astype(str)
    return pd.DataFrame({
        "note_len_chars": s.apply(len).astype(int),
        "note_len_words": s.apply(lambda x: len(x.split())).astype(int),
        "note_num_digits": s.apply(lambda x: sum(ch.isdigit() for ch in x)).astype(int),
        "note_num_neg_tokens": s.str.lower().apply(lambda x: sum(w in x for w in ["no ","denies ","without ","negative for ","not "])),
    })

# ---- 嵌入 -> PCA ----
def encode_notes(train_notes,test_notes):
    device="cuda" if (TORCH_OK and torch.cuda.is_available()) else "cpu"
    try:
        from sentence_transformers import SentenceTransformer
        model=SentenceTransformer(EMBED_MODEL_PRIMARY,device=device)
        print(f"[Emb] {EMBED_MODEL_PRIMARY} on {device}")
    except Exception:
        from sentence_transformers import SentenceTransformer
        model=SentenceTransformer(EMBED_MODEL_FALLBACK,device=device)
        print(f"[Emb] Fallback {EMBED_MODEL_FALLBACK} on {device}")
    Z_tr=model.encode(train_notes,batch_size=64,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
    Z_te=model.encode(test_notes, batch_size=64,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
    pca=PCA(n_components=EMBED_DIM_PCA,random_state=42)
    P_tr=pca.fit_transform(Z_tr); P_te=pca.transform(Z_te)
    return pd.DataFrame(P_tr,columns=[f"note_pca{i}" for i in range(P_tr.shape[1])]), \
           pd.DataFrame(P_te,columns=[f"note_pca{i}" for i in range(P_te.shape[1])])

# ---- LACE-like ----
def lace_like(df):
    los=df["los_days"].fillna(0)
    L=np.select([los<=1,los==2,los==3,(los>=4)&(los<=6),(los>=7)&(los<=13),los>=14],[1,2,3,4,5,7], default=1)
    A=df["acuity_emergent"].fillna(0).astype(int)*3
    cb=df["charlson_band"].fillna(0)
    C=np.select([cb<=0,cb==1,cb==2,cb==3,cb>=4],[0,1,2,3,5], default=0)
    ev=df["ed_visits_6m"].fillna(0)
    E=np.select([ev==0,ev==1,ev==2,ev>=3],[0,1,2,3], default=0)
    return pd.Series(L+A+C+E,name="lace_score")

# ---- LGB 表格+文本嵌入特征 ----
def assemble_lgb_features(train_df,test_df):
    for df in (train_df,test_df):
        df["discharge_weekday_int"]=df["discharge_weekday"].apply(as_weekday_int)
        df["is_weekend"]=df["discharge_weekday_int"].isin([6,7]).astype(int)
        df["is_friday"] =(df["discharge_weekday_int"]==5).astype(int)
        for c in ["los_days","ed_visits_6m","charlson_band","acuity_emergent"]:
            df[c]=pd.to_numeric(df[c],errors="coerce")
        df["los_log1p"]=np.log1p(df["los_days"].clip(lower=0))
        df["ed_visits_capped"]=df["ed_visits_6m"].clip(0,10)
        df["los_x_charlson"]=df["los_days"].fillna(0)*df["charlson_band"].fillna(0)
        df["ed_x_charlson"] =df["ed_visits_6m"].fillna(0)*df["charlson_band"].fillna(0)
        df["acuity_x_ed"]   =df["acuity_emergent"].fillna(0)*df["ed_visits_6m"].fillna(0)
    # 文本词典 / 统计 / 嵌入 / LACE
    tr_lex=lexicon_feats(train_df["note"],USE_NEGATION)
    te_lex=lexicon_feats(test_df["note"], USE_NEGATION)
    tr_lex2=lexicon_feats_extra(train_df["note"],USE_NEGATION)
    te_lex2=lexicon_feats_extra(test_df["note"], USE_NEGATION)
    tr_med=meds_followup_feats(train_df["note"])
    te_med=meds_followup_feats(test_df["note"])
    tr_stat=note_stats(train_df["note"])
    te_stat=note_stats(test_df["note"])
    P_tr,P_te=encode_notes(train_df["note"].fillna("").astype(str).tolist(),
                           test_df["note"].fillna("").astype(str).tolist())
    # 主诊断 one-hot
    for df in (train_df,test_df):
        if "primary_dx" not in df.columns: df["primary_dx"]="UNK"
    dx_all=pd.get_dummies(pd.concat([train_df["primary_dx"],test_df["primary_dx"]],axis=0).astype(str),
                          prefix="dx",drop_first=False)
    dx_tr=dx_all.iloc[:len(train_df)].reset_index(drop=True)
    dx_te=dx_all.iloc[len(train_df):].reset_index(drop=True)
    # LACE
    lace_tr=lace_like(train_df); lace_te=lace_like(test_df)
    base_cols=["los_days","los_log1p","ed_visits_6m","ed_visits_capped","charlson_band","acuity_emergent",
               "discharge_weekday_int","is_weekend","is_friday","los_x_charlson","ed_x_charlson","acuity_x_ed"]
    base_cols += ["age", "age_bucket", "sex", "insurance", "zip3",
              "pat_adm_count_all", "pat_los_mean", "pat_los_std", "pat_charlson_mean", "pat_charlson_max",
              "pat_ed_mean", "pat_ed_max", "pat_note_len_mean",
              "pat_los_days_mean_loo","pat_charlson_band_mean_loo","pat_ed_visits_6m_mean_loo","pat_note_len_chars_mean_loo",
              "te_patient","te_zip3",
             ]
    Xtr=pd.concat([train_df[base_cols].reset_index(drop=True),dx_tr,tr_lex.reset_index(drop=True),
                   tr_lex2.reset_index(drop=True),tr_med.reset_index(drop=True),tr_stat.reset_index(drop=True),
                   P_tr.reset_index(drop=True),lace_tr.reset_index(drop=True)],axis=1)
    Xte=pd.concat([test_df[base_cols].reset_index(drop=True), dx_te,te_lex.reset_index(drop=True),
                   te_lex2.reset_index(drop=True),te_med.reset_index(drop=True),te_stat.reset_index(drop=True),
                   P_te.reset_index(drop=True),lace_te.reset_index(drop=True)],axis=1)
    Xtr=Xtr.apply(pd.to_numeric,errors="coerce").fillna(0.0)
    Xte=Xte.apply(pd.to_numeric,errors="coerce").fillna(0.0)
    return Xtr,Xte,list(Xtr.columns)

# ---- LGB 训练/OOF ----
def lgb_params(seed, features):
    mono_pos={"los_log1p","ed_visits_capped","charlson_band","acuity_emergent",
              "los_x_charlson","ed_x_charlson","acuity_x_ed","lace_score"}
    mono=[1 if f in mono_pos else 0 for f in features]
    return dict(objective="binary", learning_rate=0.05, num_leaves=63, max_depth=6,
                min_data_in_leaf=120, feature_fraction=0.80, lambda_l2=5.0,
                n_estimators=2200, subsample=0.9, subsample_freq=1,
                random_state=seed, class_weight="balanced",
                n_jobs=os.cpu_count() or 4, monotone_constraints=mono, verbose=-1)

def cv_oof_lgb(X,y,groups,features):
    gkf=GroupKFold(n_splits=N_FOLDS)
    oof=np.zeros(len(y),dtype=float)
    for seed in SEEDS_LGB:
        set_seed(seed)
        for fold,(tr,va) in enumerate(gkf.split(X,y,groups=groups),1):
            clf=lgb.LGBMClassifier(**lgb_params(seed,features))
            clf.fit(X.iloc[tr],y[tr],
                    eval_set=[(X.iloc[va],y[va])],
                    eval_metric="auc",
                    callbacks=[lgb.early_stopping(stopping_rounds=200,verbose=False)])
            oof[va]+=clf.predict_proba(X.iloc[va])[:,1]
    oof/=len(SEEDS_LGB)
    return oof

def fit_lgb_full(X,y,Xte,features):
    prob=np.zeros(len(Xte),dtype=float)
    for seed in SEEDS_LGB:
        set_seed(seed)
        clf=lgb.LGBMClassifier(**lgb_params(seed,features))
        clf.fit(X,y)
        prob+=clf.predict_proba(Xte)[:,1]
    return prob/len(SEEDS_LGB)

# ---- TF-IDF（word+char） ----
def build_tfidf(train_texts,test_texts):
    v_word=TfidfVectorizer(ngram_range=(1,2),max_features=MAX_FEAT_WORD,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    v_char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),max_features=MAX_FEAT_CHAR,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    Xw_tr=v_word.fit_transform(train_texts); Xw_te=v_word.transform(test_texts)
    Xc_tr=v_char.fit_transform(train_texts); Xc_te=v_char.transform(test_texts)
    Xtr=hstack([Xw_tr,Xc_tr]); Xte=hstack([Xw_te,Xc_te])
    names_word=[f"w_{t}" for t in v_word.get_feature_names_out()]
    names_char=[f"c_{t}" for t in v_char.get_feature_names_out()]
    feat_names=sanitize_names([*names_word,*names_char])
    return Xtr,Xte,feat_names

def cv_oof_tfidf_lr(X_sparse,y,groups,C=1.0,penalty="l2"):
    gkf=GroupKFold(n_splits=N_FOLDS)
    oof=np.zeros(len(y),dtype=float)
    for fold,(tr,va) in enumerate(gkf.split(np.arange(X_sparse.shape[0]),y,groups=groups),1):
        clf=LogisticRegression(solver="saga",penalty=penalty,C=C,max_iter=500,
                               class_weight="balanced",n_jobs=os.cpu_count() or 4)
        clf.fit(X_sparse[tr],y[tr])
        oof[va]=clf.predict_proba(X_sparse[va])[:,1]
    return oof

def fit_tfidf_lr_full(X_sparse,y,X_sparse_te,C=1.0,penalty="l2"):
    clf=LogisticRegression(solver="saga",penalty=penalty,C=C,max_iter=600,
                           class_weight="balanced",n_jobs=os.cpu_count() or 4)
    clf.fit(X_sparse,y)
    probs=clf.predict_proba(X_sparse_te)[:,1]
    return clf, probs

# ---- NB-SVM log-count ratio 分数（3 列），注入 LGB 特征 ----
def nbsvm_scores(X_tr, X_te, y):
    pos = y==1; neg = ~pos
    count_pos = 1 + X_tr[pos].sum(axis=0)
    count_neg = 1 + X_tr[neg].sum(axis=0)
    p_pos = np.asarray(count_pos / count_pos.sum()).ravel()
    p_neg = np.asarray(count_neg / count_neg.sum()).ravel()
    r = np.log((p_pos + 1e-12) / (p_neg + 1e-12))
    s_tr = np.asarray(X_tr.dot(r)).ravel()
    s_te = np.asarray(X_te.dot(r)).ravel()
    return s_tr, s_te

# ---- presence 注入 ----
def topk_presence_from_lr(clf, X_tr, X_te, feat_names, k=400, min_df=5):
    coef = np.ravel(clf.coef_)
    idx_sorted = np.argsort(np.abs(coef))[::-1]
    df_counts = np.array((X_tr[:, idx_sorted] > 0).sum(axis=0)).ravel()
    keep_mask = df_counts >= min_df
    idx_filtered = idx_sorted[keep_mask][:k]
    selected = [feat_names[i] for i in idx_filtered]
    tr_bin = (X_tr[:, idx_filtered] > 0).astype(np.int8).toarray()
    te_bin = (X_te[:, idx_filtered] > 0).astype(np.int8).toarray()
    colnames = [f"ngram_top_{j}_{nm}" for j, nm in enumerate(sanitize_names(selected))]
    df_tr = pd.DataFrame(tr_bin, columns=colnames)
    df_te = pd.DataFrame(te_bin, columns=colnames)
    return df_tr, df_te, selected

# ---- 等值回归校准 ----
def isotonic_fit_transform(y_true, oof_prob, test_prob):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_prob, y_true)
    return iso.transform(oof_prob), iso.transform(test_prob), iso

# ---- 融合 ----
def prob_blend(p1, p2, alpha): return alpha*p1 + (1-alpha)*p2

# ---- 6 段: Charlson(low/mid/high) × Weekend(0/1) ----
def segment18_keys(primary_dx, cb, wkend):
    dx = pd.Series(primary_dx).fillna("UNK").astype(str)
    cb = pd.Series(cb).fillna(0)
    band = np.where(cb<=1,"low",np.where(cb>=4,"high","mid"))
    wk = np.where(pd.Series(wkend).astype(int)==1, "wknd", "wkdy")
    return np.char.add(np.char.add(np.char.add(dx.values.astype(str), "_"), band.astype(str)), "_" + wk.astype(str))

def segmented_thresholds_6(y, p, seg_keys, grid=THRESH_GRID, sample_weight=None):
    thr = {}; pred = np.zeros_like(y, dtype=int); counts={}
    uniq = np.unique(seg_keys)
    # 先求全局阈值（用于收缩）
    thr_g,_ = best_f1_threshold(y, p, grid, sample_weight=sample_weight)
    for seg in uniq:
        idx = (seg_keys==seg)
        counts[seg] = int(idx.sum())
        if counts[seg]==0:
            thr[seg]=thr_g; continue
        w = None if sample_weight is None else np.asarray(sample_weight)[idx]
        t_seg,_ = best_f1_threshold(y[idx], p[idx], grid, sample_weight=w)
        thr[seg]=t_seg
        pred[idx] = (p[idx] >= t_seg).astype(int)
    f1 = f1_score(y, pred, average="macro", sample_weight=sample_weight)
    return thr, thr_g, counts, f1, pred

def shrink_thresholds(thr_map, counts_map, thr_global, k=200):
    # James–Stein 风格收缩：t' = (n/(n+k))*t + (k/(n+k))*t_global
    out={}
    for seg,t in thr_map.items():
        n = counts_map.get(seg,0)
        w = n/(n+k)
        out[seg] = float(w*t + (1-w)*thr_global)
    return out

def apply_segment_thresholds(p, seg_keys, thr_map, default_thr=0.5):
    pred = np.zeros(len(p), dtype=int)
    for seg,thr in thr_map.items():
        idx = (seg_keys==seg); 
        if idx.sum()>0:
            pred[idx] = (p[idx] >= thr).astype(int)
    # 若有没覆盖的 seg（理论上不会），用默认阈值
    miss = ~(np.isin(seg_keys, list(thr_map.keys())))
    if miss.any():
        pred[miss] = (p[miss] >= default_thr).astype(int)
    return pred

def search_alpha_segment6_with_shrink(y, p1, p2, seg_keys, alphas=ALPHAS, grid=THRESH_GRID, k=SHRINK_K, sample_weight=None):
    best={"alpha":0.5,"thr_map":{}, "thr_map_shrink":{}, "thr_global":0.5, "f1_plain":-1.0, "f1_shrink":-1.0}
    for a in alphas:
        mix = prob_blend(p1,p2,a)
        thr_map, thr_g, counts, f1_plain, _ = segmented_thresholds_6(y, mix, seg_keys, grid, sample_weight)
        thr_map_shrink = shrink_thresholds(thr_map, counts, thr_g, k=k)
        # 计算收缩后的 OOF F1
        pred_sh = apply_segment_thresholds(mix, seg_keys, thr_map_shrink, default_thr=thr_g)
        f1_sh = f1_score(y, pred_sh, average="macro", sample_weight=sample_weight)
        if f1_sh > best["f1_shrink"]:
            best={"alpha":float(a),"thr_map":thr_map,"thr_map_shrink":thr_map_shrink,
                  "thr_global":float(thr_g),"f1_plain":float(f1_plain),"f1_shrink":float(f1_sh)}
    return best

# ============================ 主流程 ============================
def main():
    # 1) 读取 & 合并
    train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); notes=load_notes_to_df(NOTES_JSON)
    for df in (train,test,notes): df.columns=[c.strip() for c in df.columns]
    train=train.merge(notes,on="admission_id",how="left")
    test =test.merge(notes, on="admission_id",how="left")

    if USE_PATIENT_TABLE:
        train, test = add_patients(train, test, PATIENTS_CSV)
        print(f"[Patient] Added patient-level features from {PATIENTS_CSV}")

    if USE_PATIENT_HISTORY:
        train, test = add_patient_history_features(train, test)
        print("[History] Added patient history features (admission counts, means/stds, LOO means)")

    if USE_TARGET_ENCODING:
        y = train["readmit_30d"].astype(int).values
        train["te_patient"], test["te_patient"] = kfold_target_encode(
            train, test, "patient_id", y, n_splits=TE_FOLDS, seed=TE_SEED, smooth=TE_SMOOTH_PATIENT
        )
        train["te_zip3"], test["te_zip3"] = kfold_target_encode(
            train, test, "zip3", y, n_splits=TE_FOLDS, seed=TE_SEED, smooth=TE_SMOOTH_ZIP3
        )
        print("[TE] Added target encoding for patient_id and zip3")

    assert "readmit_30d" in train.columns and "patient_id" in train.columns
    y=train["readmit_30d"].astype(int).values
    groups=train["patient_id"].astype(str).values

    # 2) LGB 特征（与 v21 一致）
    Xtr_lgb, Xte_lgb, feat_names = assemble_lgb_features(train.copy(), test.copy())
    print(f"[LGB] base X_train={Xtr_lgb.shape}, X_test={Xte_lgb.shape}, features={len(feat_names)}")

    # 3) TF-IDF（word+char 组合）
    texts_tr=train["note"].fillna("").astype(str).tolist()
    texts_te=test["note"].fillna("").astype(str).tolist()
    Xtr_tfidf, Xte_tfidf, tfidf_feat_names = build_tfidf(texts_tr, texts_te)

    # 4) NB-SVM 分数（3 列）：word、char、combined -> 注入到 LGB 特征
    # 为了严格贴近 v21，这里分别再建 word/char（代价很小，相对嵌入）
    v_word=TfidfVectorizer(ngram_range=(1,2),max_features=MAX_FEAT_WORD,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    v_char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),max_features=MAX_FEAT_CHAR,
                           min_df=MIN_DF_TFIDF,strip_accents="unicode",lowercase=True,sublinear_tf=True)
    Xw_tr=v_word.fit_transform(texts_tr); Xw_te=v_word.transform(texts_te)
    Xc_tr=v_char.fit_transform(texts_tr); Xc_te=v_char.transform(texts_te)
    def _nb(Xtr, Xte, y):
        pos = y==1; neg = ~pos
        count_pos = 1 + Xtr[pos].sum(axis=0)
        count_neg = 1 + Xtr[neg].sum(axis=0)
        p_pos = np.asarray(count_pos / count_pos.sum()).ravel()
        p_neg = np.asarray(count_neg / count_neg.sum()).ravel()
        r = np.log((p_pos + 1e-12) / (p_neg + 1e-12))
        return np.asarray(Xtr.dot(r)).ravel(), np.asarray(Xte.dot(r)).ravel()
    nbw_tr, nbw_te = _nb(Xw_tr, Xw_te, y)
    nbc_tr, nbc_te = _nb(Xc_tr, Xc_te, y)
    nba_tr, nba_te = _nb(Xtr_tfidf, Xte_tfidf, y)
    Xtr_lgb["nbsvm_word"] = nbw_tr; Xte_lgb["nbsvm_word"] = nbw_te
    Xtr_lgb["nbsvm_char"] = nbc_tr; Xte_lgb["nbsvm_char"] = nbc_te
    Xtr_lgb["nbsvm_all"]  = nba_tr; Xte_lgb["nbsvm_all"]  = nba_te

    # 5) TF-LR（与 v21 一致）
    print("[TF] OOF TF-IDF(LogReg L2, C=1.0) ...")
    oof_tfidf = cv_oof_tfidf_lr(Xtr_tfidf, y, groups, C=LR_C, penalty=LR_PENALTY)
    tfidf_clf_full, pte_tfidf_raw = fit_tfidf_lr_full(Xtr_tfidf, y, Xte_tfidf, C=LR_C, penalty=LR_PENALTY)

    # 6) presence 注入（k=450）
    ngram_tr, ngram_te, sel_names = topk_presence_from_lr(tfidf_clf_full, Xtr_tfidf, Xte_tfidf,
                                                          tfidf_feat_names, k=TOP_K_NGRAMS, min_df=PRES_MIN_DF)
    Xtr_lgb_aug = pd.concat([Xtr_lgb.reset_index(drop=True), ngram_tr.reset_index(drop=True)], axis=1)
    Xte_lgb_aug = pd.concat([Xte_lgb.reset_index(drop=True), ngram_te.reset_index(drop=True)], axis=1)
    Xtr_lgb_aug = sanitize_df_columns(Xtr_lgb_aug)
    Xte_lgb_aug.columns = Xtr_lgb_aug.columns.tolist()
    feat_names_aug = Xtr_lgb_aug.columns.tolist()
    print(f"[LGB+Ngram] X_train={Xtr_lgb_aug.shape}, X_test={Xte_lgb_aug.shape}")

    # 7) LGB OOF & TEST 概率（多种子小集成）
    print("[CV] OOF LGB(AUG) ...")
    oof_lgb = cv_oof_lgb(Xtr_lgb_aug, y, groups, feat_names_aug)
    pte_lgb_raw = fit_lgb_full(Xtr_lgb_aug, y, Xte_lgb_aug, feat_names_aug)

    # 8) 两模型做 isotonic 校准
    print("[Calib] Isotonic for LGB & TF ...")
    oof_lgb_cal, pte_lgb_cal, iso_lgb = isotonic_fit_transform(y, oof_lgb, pte_lgb_raw)
    oof_tfidf_cal, pte_tfidf_cal, iso_tf = isotonic_fit_transform(y, oof_tfidf, pte_tfidf_raw)

    # 9) 在 OOF 上优化融合 + 阈值（全局 & 6 段，不带权重）
    # 9.1 先找全局最优（作为对照 & 收缩锚点）
    best_global = {"alpha":0.5,"thr":0.5,"f1":-1.0}
    for a in ALPHAS:
        mix = prob_blend(oof_lgb_cal, oof_tfidf_cal, a)
        t, f1 = best_f1_threshold(y, mix, THRESH_GRID)
        if f1 > best_global["f1"]:
            best_global = {"alpha":float(a), "thr":float(t), "f1":float(f1)}

    # 9.2 6 段（不收缩）与 6 段（收缩）共同搜索 alpha，返回两种评分做对比
    seg18_tr = segment18_keys(train["primary_dx"], Xtr_lgb_aug["charlson_band"], Xtr_lgb_aug["is_weekend"])
    best_seg6 = search_alpha_segment6_with_shrink(
        y, oof_lgb_cal, oof_tfidf_cal, seg18_tr,
        alphas=ALPHAS, grid=THRESH_GRID, k=SHRINK_K, sample_weight=None
    )

    print(f"[Global] alpha={best_global['alpha']:.2f}, thr={best_global['thr']:.3f}, OOF Macro-F1={best_global['f1']:.4f}")
    print(f"[Seg6-plain]  alpha={best_seg6['alpha']:.2f}, OOF Macro-F1={best_seg6['f1_plain']:.4f}")
    print(f"[Seg6-shrink] alpha={best_seg6['alpha']:.2f}, OOF Macro-F1={best_seg6['f1_shrink']:.4f}")
    print(f"              thr_global={best_seg6['thr_global']:.3f}")
    print(f"              thr_map(shrink)={best_seg6['thr_map_shrink']}")

    # 10) 生成三份提交
    # 10.1 全局
    pte_blend_g = prob_blend(pte_lgb_cal, pte_tfidf_cal, best_global["alpha"])
    pred_global = (pte_blend_g >= best_global["thr"]).astype(int)
    sub_g = test[["admission_id"]].copy(); sub_g["readmit_30d"] = pred_global
    sub_g.to_csv("submission_v25_global.csv", index=False)

    # 10.2 6 段（不收缩）
    pte_blend_s = prob_blend(pte_lgb_cal, pte_tfidf_cal, best_seg6["alpha"])
    seg6_te = segment6_keys(Xte_lgb_aug["charlson_band"], Xte_lgb_aug["is_weekend"])
    pred_seg_plain = apply_segment_thresholds(pte_blend_s, seg6_te, best_seg6["thr_map"], default_thr=best_seg6["thr_global"])
    sub_sp = test[["admission_id"]].copy(); sub_sp["readmit_30d"] = pred_seg_plain
    sub_sp.to_csv("submission_v25_segment6_plain.csv", index=False)

    # 10.3 6 段（收缩）
    pred_seg_sh = apply_segment_thresholds(pte_blend_s, seg6_te, best_seg6["thr_map_shrink"], default_thr=best_seg6["thr_global"])
    sub_ss = test[["admission_id"]].copy(); sub_ss["readmit_30d"] = pred_seg_sh
    sub_ss.to_csv("submission_v25_segment6_shrink.csv", index=False)

    # 11) 简要 OOF 报告（6 段收缩）
    mix_oof = prob_blend(oof_lgb_cal, oof_tfidf_cal, best_seg6["alpha"])
    y_pred_oof_sh = apply_segment_thresholds(mix_oof, seg18_tr, best_seg6["thr_map_shrink"], default_thr=best_seg6["thr_global"])
    print("\n=== OOF Classification Report (Seg6-shrink) ===")
    print(classification_report(y, y_pred_oof_sh, digits=4))

    print("Saved:")
    print(" - submission_v25_segment6_shrink.csv")
    print(" - submission_v25_segment6_plain.csv")
    print(" - submission_v25_global.csv")

if __name__ == "__main__":
    if not LGB_OK:
        raise ImportError("LightGBM required. Install: pip install lightgbm")
    main()


[Patient] Added patient-level features from D:/AgentDs/agent_ds_healthcare/patients.csv
[History] Added patient history features (admission counts, means/stds, LOO means)
[TE] Added target encoding for patient_id and zip3


No sentence-transformers model found with name emilyalsentzer/Bio_ClinicalBERT. Creating a new one with mean pooling.


[Emb] emilyalsentzer/Bio_ClinicalBERT on cuda


Batches: 100%|██████████| 79/79 [00:07<00:00,  9.89it/s]


[LGB] base X_train=(5000, 138), X_test=(5000, 138), features=138
[TF] OOF TF-IDF(LogReg L2, C=1.0) ...
[LGB+Ngram] X_train=(5000, 591), X_test=(5000, 591)
[CV] OOF LGB(AUG) ...
[Calib] Isotonic for LGB & TF ...
[Global] alpha=0.90, thr=0.475, OOF Macro-F1=0.9088
[Seg6-plain]  alpha=0.98, OOF Macro-F1=0.9126
[Seg6-shrink] alpha=0.98, OOF Macro-F1=0.9106
              thr_global=0.465
              thr_map(shrink)={np.str_('DiabetesComp_high_wkdy'): 0.46601593625498006, np.str_('DiabetesComp_high_wknd'): 0.4256896551724138, np.str_('DiabetesComp_low_wkdy'): 0.38208754208754214, np.str_('DiabetesComp_low_wknd'): 0.4671988795518207, np.str_('DiabetesComp_mid_wkdy'): 0.38163686382393397, np.str_('DiabetesComp_mid_wknd'): 0.465, np.str_('HF_high_wkdy'): 0.5714141414141415, np.str_('HF_high_wknd'): 0.4001423487544484, np.str_('HF_low_wkdy'): 0.46787234042553194, np.str_('HF_low_wknd'): 0.4875163398692811, np.str_('HF_mid_wkdy'): 0.5285931558935362, np.str_('HF_mid_wknd'): 0.46840255591054314,

# Submission

In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 1, "submission_v25_global.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 2!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 0.9058 (Macro-F1)
✅ Validation passed
✅ Submission successful!
   📊 Score: 0.9058
   📏 Metric: Macro-F1
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 2!
